### 0. Schema
<img src="../figures/schema.png" width="70%">  

### 1. DataFrames Initialization

In [278]:
import numpy as np
import pandas as pd

In [279]:
orders = pd.read_csv(r'..\data\raw\olist_orders_dataset.csv', index_col='order_id', 
                        parse_dates=[3, 4, 5, 6, 7], date_format="%Y-%m-%d %H:%M:%S")
items = pd.read_csv(r'..\data\raw\olist_order_items_dataset.csv', index_col='order_id', 
                    parse_dates=['shipping_limit_date'], date_format="%Y-%m-%d %H:%M:%S")
sellers = pd.read_csv(r'..\data\raw\olist_sellers_dataset.csv', index_col='seller_id')
geolocations = pd.read_csv(r'..\data\raw\olist_geolocation_dataset.csv', index_col='geolocation_zip_code_prefix')
customers = pd.read_csv(r'..\data\raw\olist_customers_dataset.csv', index_col='customer_id')
reviews = pd.read_csv(r'..\data\raw\olist_order_reviews_dataset.csv', index_col='order_id', 
                        parse_dates=['review_creation_date', 'review_answer_timestamp'], 
                        date_format="%Y-%m-%d %H:%M:%S")
payments = pd.read_csv(r'..\data\raw\olist_order_payments_dataset.csv', index_col='order_id')
products = pd.read_csv(r'..\data\raw\olist_products_dataset.csv', index_col='product_id')
cat_trans = pd.read_csv(r'..\data\raw\product_category_name_translation.csv', 
                        index_col='product_category_name') 

### 2. orders 

Granularity:  
One row represents one order.

Index:  
order_id

In [280]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99441 entries, e481f51cbdc54678b7cc49136f2d6af7 to 66dea50a8b16d9b4dee7af250b4be1a5
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   customer_id                    99441 non-null  object        
 1   order_status                   99441 non-null  object        
 2   order_purchase_timestamp       99441 non-null  datetime64[ns]
 3   order_approved_at              99281 non-null  datetime64[ns]
 4   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 5   order_delivered_customer_date  96476 non-null  datetime64[ns]
 6   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(2)
memory usage: 6.1+ MB


In [281]:
orders.index.duplicated().sum()

np.int64(0)

**Observation**

- The table contains approximately 100,000 orders. The unique order_id index indicates that the table stores one record per order. Orders containing multiple products are represented separately in the order items table.
- Four timestamp columns (`order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`) describe different stages of the order fulfillment process.
- The number of non-null values decreases as the order progresses through the fulfillment process.
- The number of non-null values of `order_purchase_timestamp`  and `order_estimated_delivery_date` are the same.
- Both estimated and actual delivery dates are available, making it possible to evaluate delivery performance.

**Questions**

- Why are timestamps missing in later stages of the order lifecycle?
- How should incomplete orders be handled in subsequent analyses?
- Does delivery performance influence customer review scores?
- Can a single order involve multiple products, therefore multiple sellers, and if so, how is the delivery date recorded in the orders table?

**Working Hypotheses**

- Not every purchased order is approved. Some approved orders may not reach the carrier or the customer because of cancellations, payment failures, logistics issues, or other exceptional events. Missing timestamps in later stages may therefore represent orders that did not complete the entire fulfillment process. 

- `order_estimated_delivery_date` are generated after purchasing an order.

- The delivery timestamp stored in the orders table may correspond to the completion of the entire order rather than individual items, which could complicate analyses when an order contains products from multiple sellers.

### 3. items

Granularity:  
One row represents one item in an order.

Index:  
order_id

In [282]:
items.info()

<class 'pandas.core.frame.DataFrame'>
Index: 112650 entries, 00010242fe8c5a6d1ba2dd792cb16214 to fffe41c64501cc87c801fd61db3f6244
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_item_id        112650 non-null  int64         
 1   product_id           112650 non-null  object        
 2   seller_id            112650 non-null  object        
 3   shipping_limit_date  112650 non-null  datetime64[ns]
 4   price                112650 non-null  float64       
 5   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(2)
memory usage: 6.0+ MB


In [283]:
items.index.duplicated().sum()

np.int64(13984)

**Observation** 
- The table contains 112,650 order items, which is greater than the number of orders because each row represents a single item within an order.
- The DataFrame index (`order_id`) is therefore non-unique, reflecting a one-to-many relationship between orders and order items. Selecting by `order_id` may return multiple rows.
- Each row represents one purchased item rather than an entire order. The table does not contain a quantity column; each row represents an item within an order.

**Initial Questions**

- How many items does a typical order contain?
- Are shipping costs proportional to the number of items or their total price?

### 4. sellers
Granularity:  
One row represents one seller.

Index:  
seller_id

In [284]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3095 entries, 3442f8959a84dea7ee197c632cb2df15 to 9e25199f6ef7e7c347120ff175652c3b
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_zip_code_prefix  3095 non-null   int64 
 1   seller_city             3095 non-null   object
 2   seller_state            3095 non-null   object
dtypes: int64(1), object(2)
memory usage: 96.7+ KB


**Observations**

- The table contains 3,095 sellers.
- Each seller is uniquely identified by `seller_id`, which is used as the DataFrame index.
- Only geographic information (ZIP code prefix, city, and state) is provided. No seller-level business metrics, such as registration date or seller category, are available.

**Initial Questions**

- How are sellers geographically distributed across Brazil?
- How many orders are associated with each seller?
- Does seller location relate to delivery performance or customer satisfaction after merging with other tables?

### 5. geolocations

Granularity:  
One row represents one location.

Index:  
zip_code_prefix

In [305]:
geolocations.columns = pd.Index(['lat', 'lng', 'city', 'state'])
geolocations.index.name = 'zip_code_prefix'
geolocations.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000163 entries, 1037 to 99950
Data columns (total 4 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   lat     1000163 non-null  float64
 1   lng     1000163 non-null  float64
 2   city    1000163 non-null  object 
 3   state   1000163 non-null  object 
dtypes: float64(2), object(2)
memory usage: 70.4+ MB


In [306]:
geolocations.index.duplicated().sum()

np.int64(981148)

Question: Do duplicated zip code prefixes correspond to duplicated latitude–longitude pairs?

In [307]:
geolocations.index.duplicated() == geolocations[['lat', 'lng']].duplicated()

zip_code_prefix
1037      True
1046      True
1046     False
1041      True
1035      True
         ...  
99950    False
99900     True
99950     True
99980    False
99950     True
Length: 1000163, dtype: bool

Question: Which field provides a more granular geographical grouping: zip_code_prefix or city?

In [308]:
pd.Series({
    "Unique zip_code_prefix": geolocations.index.nunique(),
    "Unique cities": geolocations["city"].nunique()
})

Unique zip_code_prefix    19015
Unique cities              8011
dtype: int64

**Observations**

- The table contains over one million geographic records. The index (`zip_code_prefix`) is not unique, indicating that multiple records may correspond to the same ZIP code prefix.
- The duplication patterns of `zip_code_prefix` and latitude–longitude pairs are different, indicating that their relationship is not one-to-one.
- ZIP code prefix provides a more detailed grouping than city in this dataset, although the relationship between ZIP codes and coordinates is not one-to-one.

**Initial Questions**

- What is the relationship between ZIP code prefixes and geographic coordinates? Is the relationship many-to-many, one-to-many, or many-to-one?
- How should geographic information be assigned to an order when a ZIP code prefix corresponds to multiple records in the geolocation table?
- Is each ZIP code prefix associated with a unique city, state, and geographic coordinate, or are conflicting records present?
- If multiple matches exist after merging, what strategy should be adopted to obtain a single representative location?
- At what geographic level should analyses be performed (state, city, ZIP code prefix, or geographic coordinates)?

### 6. customers

Granularity:  
Granularity: One row represents the customer information recorded for an order.

Index:  
customer_id

In [289]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99441 entries, 06b8999e2fba1a1fbc88172c00ba8bc7 to 274fa6071e5e17fe303b9748641082c8
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_unique_id        99441 non-null  object
 1   customer_zip_code_prefix  99441 non-null  int64 
 2   customer_city             99441 non-null  object
 3   customer_state            99441 non-null  object
dtypes: int64(1), object(3)
memory usage: 3.8+ MB


In [291]:
pd.Series({
    "Duplicated customers.index": customers.index.duplicated().sum(),
    "Duplicated customer_unique_id": customers["customer_unique_id"].duplicated().sum()
})

Duplicated customers.index          0
Duplicated customer_unique_id    3345
dtype: int64

Question: Is there a one-to-one relationship between orders.customer_id and customers.customer_id?

In [292]:
[orders.index.is_unique, customers.index.is_unique, set(orders["customer_id"]) == set(customers.index)]


[True, True, True]

**Observations**
- Two customer identifiers are provided: `customer_id` (the DataFrame index) and `customer_unique_id`. The index (`customer_id`) is unique, while `customer_unique_id` is not. The number of customer records matches the number of orders, and `customer_id` is also unique in the `orders` DataFrame.
- Each customer record is associated with a ZIP code prefix, city, and state, providing a recorded geographic location for each customer record.

**Initial Questions**

- What is the distinction between `customer_id` and `customer_unique_id`? Which identifier should be used in customer-level analyses?
- Does the recorded customer location represent the customer's registered address or the delivery location of an order? If they differ, where is the delivery location recorded?

**Working Hypotheses**

- `customer_id` may identify a customer record associated with an individual order, whereas `customer_unique_id` may identify the underlying customer across multiple purchases. This hypothesis is motivated by the equal number of customer_id records in the `customers` and `orders` DataFrames, the uniqueness of `customer_id` in both tables, and the non-uniqueness of `customer_unique_id`.
- The recorded customer location may represent a default or registered address rather than the delivery destination of every order. If customers are able to ship orders to different addresses, additional delivery location information may exist elsewhere in the dataset or may not be available.

### 7. reviews

Granularity:  
One row represents one review record associated with an order.

Index:  
order_id

In [293]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99224 entries, 73fc7af87114b39712e6da79b0a377eb to 90531360ecb1eec2a1fbb265a0db0508
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  object        
 1   review_score             99224 non-null  int64         
 2   review_comment_title     11568 non-null  object        
 3   review_comment_message   40977 non-null  object        
 4   review_creation_date     99224 non-null  datetime64[ns]
 5   review_answer_timestamp  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(3)
memory usage: 5.3+ MB


Question: Can duplicated order IDs in the reviews dataframe be explained by orders containing multiple items?

In [294]:
pd.Series({
'Duplicated order_ID in this dataframe': reviews.index.duplicated().sum(),
'Duplicated order_ID in the items dataframe': items.index.duplicated().sum()
})

Duplicated order_ID in this dataframe           551
Duplicated order_ID in the items dataframe    13984
dtype: int64

Question: How do review records differ when they share the same order ID?

In [295]:
reviews.loc[reviews.index.duplicated(keep=False)].sort_index()

,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
order_id,,,,,,
0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12
0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25,2017-08-29 21:45:57
013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,NaN,NaN,2017-12-30,2018-01-02 10:54:06
...,...,...,...,...,...,...
ff763b73e473d03c321bcd5a053316e8,5e78482ee783451be6026e5cf0c72de1,3,NaN,Não sei que haverá acontecido os demais chegaram,2017-11-18,2017-11-18 09:02:48
ff850ba359507b996e8b2fbb26df8d03,39de8ad3a1a494fc68cc2d5382f052f4,5,NaN,Envio rapido... Produto 100%,2017-08-16,2017-08-17 11:56:55
ff850ba359507b996e8b2fbb26df8d03,80f25f32c00540d49d57796fb6658535,5,NaN,"Envio rapido, produto conforme descrito no anu...",2017-08-22,2017-08-25 11:40:22


Question: Is there evidence that some review records are automatically generated after a fixed period?

In [296]:
time_lapse = reviews['review_answer_timestamp'] - reviews['review_creation_date']
time_lapse.value_counts().sort_values(ascending=False)

1 days 00:44:33     7
0 days 23:05:28     6
1 days 00:36:16     6
1 days 00:43:35     6
1 days 10:53:21     6
                   ..
19 days 22:17:07    1
1 days 17:28:41     1
1 days 17:10:12     1
2 days 19:49:33     1
1 days 12:59:13     1
Name: count, Length: 82090, dtype: int64

**Observations**

- Only a minority of reviews contain a comment message, and an even smaller subset contains a comment title.
- A preliminary inspection suggests that most comment texts are written in Portuguese, so translation or multilingual text processing may be required for later textual analysis.
- The number of duplicated order IDs in reviews differs from that in order_items, suggesting that multiple products per order cannot fully explain duplicated review records.
- `review_score` is an integer rating from 1 to 5, where 5 represents the highest level of satisfaction.
- Some orders are associated with multiple reviews, which are generally created or answered at different timestamps.

**Initial Questions**

- How can review comments be translated and analyzed to extract sentiment, topics, and specific service issues?
- Do orders without customer feedback receive a default review score, or are they excluded from the review table?
- Why do some orders have multiple review records? Do they correspond to different items, multiple review submissions, or later updates to an earlier review?

**Working Hypotheses**

- Multiple review records associated with the same order may be related to multiple items within the order, although the review table itself contains no item-level identifier to verify this directly.

- The distribution of `time_lapse` shows a large number of unique intervals. The most frequent interval occurring only seven times, while over 82,000 different intervals exist. This does not provide evidence of a systematic automatic generation process.

- Since the number of duplicated `order_id` values differs between the `reviews` and `items` DataFrames, the relationship is unlikely to be explained solely by the number of purchased items. Further investigation is required.

### 8. payments


Granularity:  
One row represents one payment record.

Index:  
order_id

In [297]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 103886 entries, b81ef226f3fe1789b1e8b2acac839d17 to 28bbae6599b09d39ca406b747b6632b1
Data columns (total 4 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   payment_sequential    103886 non-null  int64  
 1   payment_type          103886 non-null  object 
 2   payment_installments  103886 non-null  int64  
 3   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(1)
memory usage: 4.0+ MB


In [298]:
payments.index.duplicated().sum()

np.int64(4446)

Question: How do payment records differ when they share the same order ID?

In [299]:
payments[payments.index.duplicated(keep=False)].sort_index()

,payment_sequential,payment_type,payment_installments,payment_value
order_id,,,,
0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
...,...,...,...,...
ffa1dd97810de91a03abd7bd76d2fed1,2,voucher,1,418.73
ffa39020fe7c8a3e907320e1bec4b985,1,credit_card,1,7.13
ffa39020fe7c8a3e907320e1bec4b985,2,voucher,1,64.01


Question: Can duplicated order IDs in the payment dataframe be explained by payment sequential is larger than 1?

In [311]:
payments[payments['payment_sequential'] > 1].index

Index(['5cfd514482e22bc992e7693f0e3e8df7', '3689194c14ad4e2e7361ebd1df0e77b0',
       '21b8b46679ea6482cbf911d960490048', 'ea9184ad433a404df1d72fa0a8764232',
       '82ffe097d8ddbf319a523b9bbe7725d5', '487c1451b8fd7347d0e80e5aca887e91',
       '65c863d90026e9dd4888e28d2135c983', '591083bc42b589c7052118aa83118e76',
       '79b63d1f9cf0257b33b04fbfb0847cba', '332c6742772f2df936696b6512b10edb',
       ...
       '08fa8b49ff198d1332df4668087150ed', '71853944ebfe6bd5f5de0302cba14354',
       '67937bb4e0c9a4156fe47c89382d33e7', 'e2da042c42ce790c81ef3a9f666a92b6',
       '1ea11b975aea60ca92fa0559184d3d34', 'fedcd9f7ccdc8cba3a18defedd1a5547',
       'f0a5b7c94819c69d12a1c0458ec74756', 'cf101c3abd3c061ca9f78c1bbb1125af',
       'd0bff47153ef056bb4f884a2ec2f0691', '31bc09fdbd701a7a4f9b55b5955b8687'],
      dtype='object', name='order_id', length=4526)

In [314]:
multiple_payment_sequences = payments[payments['payment_sequential'] > 1].sort_index()
multiple_payment_sequences

,payment_sequential,payment_type,payment_installments,payment_value
order_id,,,,
0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
009ac365164f8e06f59d18a08045f6c4,6,voucher,1,4.17
...,...,...,...,...
ff978de32e717acd3b5abe1fb069d2b6,2,voucher,1,10.80
ff978de32e717acd3b5abe1fb069d2b6,4,voucher,1,7.68
ffa1dd97810de91a03abd7bd76d2fed1,2,voucher,1,418.73


Question: Is  payment sequential is larger than 1 always associated with different payment types?

In [301]:
multiple_payment_sequences.groupby('order_id')['payment_type'].nunique().sort_values()

order_id
40aa89b7cd7a1c649a752f6539c4a57b    1
40edd29fcc7f120710d42d0400bc8e29    1
414643c3ed26cd6d93ac777a102a3ac0    1
41932b54e17f6ae1f4be1f8886f47883    1
41b0c346ca595e5a94067d231e4106af    1
                                   ..
00b4a910f64f24dbcac04fe54088a443    2
ffa1dd97810de91a03abd7bd76d2fed1    2
ffa39020fe7c8a3e907320e1bec4b985    2
009ac365164f8e06f59d18a08045f6c4    2
0016dfedd97fc2950e388d2971d718c7    2
Name: payment_type, Length: 3039, dtype: int64

**Observations**
-  The index of the `payments` dataframe (`order_id`) has duplicates. 
- Some payment records have `payment_sequential > 1`. Since the sequence starts from 1, these orders are associated with multiple payment records. In addition, `payment_installments` can exceed 1, meaning that a payment may be split into multiple installments.
- Initial inspection suggests that credit cards and vouchers are common payment methods.

**Initial Questions**

- Under what circumstances does one order generate multiple payment records?
- Are duplicated payment records associated with multiple payment methods, multiple items within an order, or other payment processes?


**Working Hypotheses**

- Initial inspection shows that orders with multiple payment types are associated with `payment_sequential > 1`, suggesting that this is one mechanism that produces multiple payment records for a single order.

- However, duplicated payment records are also observed when all records share the same `payment_type`. 
Additional factors may contribute to multiple payment records and will be investigated by combining the payment table with other related tables in the next notebook.

### 9. products

Granularity:  
One row represents one product.

Index:  
'product_id'

In [302]:
products['product_category_name'] = products['product_category_name'].map(
                    cat_trans["product_category_name_english"])


In [303]:
products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32951 entries, 1e9e8ef04dbcff4541ed26657ea517e5 to 106392145fca363410d287a815be6de4
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_category_name       32328 non-null  object 
 1   product_name_lenght         32341 non-null  float64
 2   product_description_lenght  32341 non-null  float64
 3   product_photos_qty          32341 non-null  float64
 4   product_weight_g            32949 non-null  float64
 5   product_length_cm           32949 non-null  float64
 6   product_height_cm           32949 non-null  float64
 7   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(1)
memory usage: 2.3+ MB


In [304]:
products.index.duplicated().sum()

np.int64(0)

**Observations**

- The DataFrame's index (`product_id`) uniquely identifies each product.
- Products are classified into predefined categories with translation provided by the dataset.
- The DataFrame contains both textual metadata (name length, description length, photo count) and physical characteristics (weight and dimensions).
- This table provides product-level attributes that can be merged into transaction data if product characteristics become relevant during later analyses.

**Initial Questions**

- Do certain product categories receive better reviews or experience longer delivery times?
- How strongly are product weight and dimensions associated with freight charges?
- Does the amount of product information (description length or number of photos) influence customer satisfaction?
- How should missing product categories be handled?

### 10. Summary